In [ ]:
import tifffile
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import cv2
from skimage.measure import regionprops, label

In [ ]:
test = tifffile.imread(Path(r"z:\Bel\Farid\Paired_Dextran_Images\vm_outputs\010826_N3_flow_day_10_R_1_Merged_8channel.tif"))

In [ ]:
dextran = test[:,2,:,:]
full_segmentation = test[:,-4,:,:]
valid_region = test[:,-3,:,:]

In [ ]:
valid_dextran = dextran * (valid_region>0) * (full_segmentation>0)

In [ ]:
chip_2d = np.max(valid_dextran, axis=0)
plt.imshow(chip_2d, cmap='gray')
plt.show()

In [ ]:
labelled_device = label(chip_2d)
props = regionprops(labelled_device)

for prop in props:
    mid_x = prop.centroid[1]
    mid_y = prop.centroid[0]
    width = prop.major_axis_length
    height = prop.minor_axis_length
    orientation = prop.orientation
    corner_bl = (mid_x - (0.5*width*np.sin(orientation)), mid_y-0.5*height*np.cos(orientation))
    corner_br = (mid_x + (0.5*width*np.sin(orientation)), mid_y-0.5*height*np.cos(orientation))
    corner_tl = (mid_x - (0.5*width*np.sin(orientation)), mid_y+0.5*height*np.cos(orientation))
    corner_tr = (mid_x + (0.5*width*np.sin(orientation)), mid_y+0.5*height*np.cos(orientation))

    plt.plot([corner_bl[0], corner_br[0], corner_tr[0], corner_tl[0], corner_bl[0]],
             [corner_bl[1], corner_br[1], corner_tr[1], corner_tl[1], corner_bl[1]], 'r-')

plt.imshow(chip_2d, cmap='gray')
plt.show()

In [ ]:
final_corners = np.array([
            [0,0],
            [width-1,0],
            [width-1,height-1],
            [0,height-1]], dtype=np.float32) 

In [ ]:

M = cv2.getPerspectiveTransform(src, dst)

img_rectified = cv2.warpPerspective(
    intensity_image.astype(np.float32),
    M,
    (width, height),
    flags=cv2.INTER_LINEAR
)

mask_rectified = cv2.warpPerspective(
    mask,
    M,
    (width, height),
    flags=cv2.INTER_NEAREST
).astype(bool)

# Only include pixels belonging to original label
img_masked = np.where(mask_rectified, img_rectified, 0)

# Sum left -> right for every y
profile = img_masked.sum(axis=1)

return profile, img_rectified, mask_rectified